# 02 Modeling (ML Lead sandbox)
## Your job in this notebook
Train and compare models on the **train** and **validation** splits: try feature sets, model settings and ideas.
Your files: `src/solarcast/models.py`, `evaluate.py`, `pipeline.py`. Move what works into those files.

## Setup (run this first)
This cell makes the notebook work both in Google Colab and on a laptop. In Colab it clones (or updates) the
GitHub repo into `/content/SolarCast-Lite`; locally it finds the repo root. Then it puts `src/` on the Python path
and turns on autoreload, so edits to `src/solarcast/*.py` are picked up without restarting.
If the repo is private, add a Colab Secret named `GITHUB_TOKEN` (see `docs/COLAB_GUIDE.md`). The token is never printed.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/nashwu0117/SolarCast-Lite.git"
IN_COLAB = "google.colab" in sys.modules


def _git(args, cwd=None, secret=None, check=True):
    """Run git, hiding the token if it shows up in any output."""
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if secret:
        out = out.replace(secret, "***")
    if check and r.returncode != 0:
        raise RuntimeError("git " + args[0] + " failed:\n" + out)
    return r.returncode, out


if IN_COLAB:
    REPO = Path("/content/SolarCast-Lite")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None  # no secret: fine for a public repo
    auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
    if not REPO.exists():
        _git(["clone", auth_url, str(REPO)], secret=token)
        _git(["remote", "set-url", "origin", REPO_URL], cwd=REPO)  # keep the token out of .git/config
    elif not (REPO / ".git").exists():
        print("Using the uploaded copy in", REPO, "(not a git clone, so no pull).")
    else:
        _git(["remote", "set-url", "origin", auth_url], cwd=REPO, secret=token)
        code, out = _git(["pull", "--ff-only"], cwd=REPO, secret=token, check=False)
        _git(["remote", "set-url", "origin", REPO_URL], cwd=REPO)
        if code != 0:
            print("git pull did not fast-forward (local edits?). Continuing with the current files.\n" + out)
    os.chdir(REPO)
    if importlib.util.find_spec("gradio") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
else:
    REPO = Path.cwd().resolve()
    while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
        REPO = REPO.parent
    os.chdir(REPO)

# Editable installs do not take effect in Colab without a runtime restart, so use the path directly.
sys.path.insert(0, str(REPO / "src"))
print("Repo:", REPO, "| Colab:", IN_COLAB)

# Pick up edits to src/solarcast/*.py without restarting. Colab's IPython 7 autoreload imports the
# removed `imp` module on Python 3.12+, so fall back gracefully instead of failing the whole notebook.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception as exc:
    print(f"autoreload unavailable ({type(exc).__name__}); after editing src/, use Runtime > Restart session.")

> **Warning: do not tune on the test split.**
> Use `train` to fit and `val` to choose. The test split (Mar-Jun 2026) is looked at once, at the end, for reporting.
> If you change a setting because it improved test scores, the reported numbers are no longer honest.

In [ ]:
from solarcast import models
from solarcast.baselines import BASELINES
from solarcast.evaluate import compare, split_by_time
from solarcast.features import build_features
from solarcast.data import load_site

feat = build_features(load_site("A"))
splits = split_by_time(feat)
{k: v.shape for k, v in splits.items()}

### Train Model A and Model B, score on validation

In [ ]:
preds = {name: fn(splits["val"]) for name, fn in BASELINES.items()}
for name, fs in (("Model A", "A"), ("Model B", "B")):
    m = models.train_model(splits["train"], fs)
    preds[name] = models.predict(m, splits["val"])
compare(splits["val"], preds).round(3)

### Your experiments (validation only)
Example ideas: other model settings in `make_model`, extra features, different lags.

In [ ]:
# Example: try a different model setting on validation
from sklearn.ensemble import HistGradientBoostingRegressor

# m2 = HistGradientBoostingRegressor(max_depth=4, learning_rate=0.05, random_state=42)